# 06 · Model choice, reproducibility and limits

**Purpose:** justify the model, show what was measured, say what is not guaranteed, and list what a production version would add.  
**Decisions implemented:** D5 (replay), D7 (model), B2 criteria 6 and 8.  
**Inputs → outputs:** `outputs/bakeoff/metrics.json`, `docs/bakeoff-report.md`, `outputs/stage12_metrics.json` → tables.

*Run from the project root or from `notebooks/`; nothing is written, and no model is called (replay mode).*

In [1]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image
from ti import nbhelpers as nb
nb.setup()
print("replay mode: no network, no API key, no spend")

replay mode: no network, no API key, no spend


## Why one model, and how it was chosen (D7)

Public benchmarks for this kind of work (tone, strict JSON, verbatim quoting) were thin, and coding-agent benchmarks do not measure it, so they were used only as a weak prior.
A **bake-off on our own data** compared Claude Haiku 5.5, Claude Sonnet 5.5, GPT-6.1 Terra and Gemini 3.8 Flash through OpenRouter, with the provider pinned per model. Pinned: `google/gemini-3.8-flash` for every stage.

In [2]:
b = nb.read_json("outputs/bakeoff/metrics.json")
tot = b["totals"]
nb.md(f"Bake-off: {tot['calls']} calls, {tot['failed']} failed, total cost **${tot['total_cost_usd']:.2f}**. By model: " + ", ".join(f"{k} ${v:.2f}" for k, v in tot["by_model"].items()) + ".")
g0, h0 = b["stage0"]["gemini"], b["stage0"]["haiku"]
nb.table(pd.DataFrame([
    {"Stage 0 (140 calls each)": "schema-valid first attempt", "Haiku 5.5": f"{h0['schema_valid_first_attempt']:.1f}%", "Gemini 3.8 Flash": f"{g0['schema_valid_first_attempt']:.1f}%"},
    {"Stage 0 (140 calls each)": "verbatim quote (exact)", "Haiku 5.5": f"{h0['quote_valid_exact']:.1f}%", "Gemini 3.8 Flash": f"{g0['quote_valid_exact']:.1f}%"},
    {"Stage 0 (140 calls each)": "disagreements with title rule (run 1, of 70 calls)", "Haiku 5.5": f"{round((100 - h0['agree_title_rule']) * 0.7)}", "Gemini 3.8 Flash": f"{round((100 - g0['agree_title_rule']) * 0.7)}"},
    {"Stage 0 (140 calls each)": "run 1 vs run 2 same label", "Haiku 5.5": f"{h0['run_to_run_label_agreement']:.1f}%", "Gemini 3.8 Flash": f"{g0['run_to_run_label_agreement']:.1f}%"},
    {"Stage 0 (140 calls each)": "cost", "Haiku 5.5": f"${h0['cost_usd']:.3f}", "Gemini 3.8 Flash": f"${g0['cost_usd']:.3f}"},
]))
r12 = b["rules"]["stage12"]["stats"]
nb.table(pd.DataFrame([{"model (Stages 1 and 2, 15-call sample, draft prompts)": k, "schema": v["schema"], "quote": v["quote"], "stability": round(v["stability"], 1), "cost $": round(v["cost"], 3)} for k, v in r12.items()]))

Bake-off: 475 calls, 0 failed, total cost **$1.63**. By model: haiku $0.04, sonnet $0.90, terra $0.29, gemini $0.41.

Stage 0 (140 calls each),Haiku 5.5,Gemini 3.8 Flash
schema-valid first attempt,100.0%,100.0%
verbatim quote (exact),100.0%,98.6%
"disagreements with title rule (run 1, of 70 calls)",5,3
run 1 vs run 2 same label,100.0%,100.0%
cost,$0.040,$0.166


"model (Stages 1 and 2, 15-call sample, draft prompts)",schema,quote,stability,cost $
sonnet,100.0,100.0,98.6,0.835
terra,100.0,100.0,97.6,0.270
gemini,100.0,100.0,99.7,0.225


### The decision rule had a flaw, and it was overridden

The pre-written rule disqualified on format failures and then picked the **cheapest** model. Its "quality" score averaged schema validity, quote validity and run-to-run stability, so it measured **format, not correctness**.
Every model passed every threshold, so the rule just chose the cheapest: Haiku for Stage 0. But on two contested calls Haiku called an account-manager outreach call "support"
(the reviewer sided with Gemini on both, notebook 01). The rule could not see that. D7 overrides it for Stage 0, and the deck should say so plainly: a design flaw found in review, not hidden.

In [3]:
nb.table(pd.DataFrame([{"stage": s, "rule's winner": v["winner"], "disqualified": str(v["disqualified"]) or "none", "eligible": ", ".join(v["eligible"])} for s, v in b["rules"].items()]))

stage,rule's winner,disqualified,eligible
stage0,haiku,{},"haiku, gemini"
stage12,gemini,{},"sonnet, terra, gemini"


## Reproducibility: temperature 0 does not guarantee identical output

In [4]:
rows = []
for k, v in b["determinism"].items():
    rows.append({"stage / model": k, "transcripts re-run 3 times": v["of"], "byte-identical JSON": v["identical_json_count"]})
nb.table(pd.DataFrame(rows))

stage / model,transcripts re-run 3 times,byte-identical JSON
stage0/haiku,3,2
stage0/gemini,3,3
stage2/sonnet,3,0
stage2/terra,3,0
stage2/gemini,3,2


Claude and GPT models do not list `temperature` as a supported parameter, and even the model that does (Gemini) repeated exactly on only some transcripts. So a live re-run will not reproduce a run exactly. **The guarantee is the committed cache of raw responses**
(replay mode), shown in notebook 00: each stage replays with 0 differences. The cache key includes the prompt text, model, provider, parameters and input, so any change re-runs the stage for every call (one method over time).

In [5]:
m = nb.read_json("outputs/stage12_metrics.json")
bake = b["totals"]["total_cost_usd"]; s1 = m["stage1"]["cost"]; s2 = m["stage2"]["cost"]
runs = m["stage1_stability"]["cost_run2"] + m["stage2_stability"]["cost_run2"]
nb.table(pd.DataFrame([{"item": "bake-off (4 models, 475 calls)", "USD": round(bake, 2)}, {"item": "Stage 1 over 100 calls", "USD": round(s1, 2)},
                       {"item": "Stage 2 over 100 calls", "USD": round(s2, 2)}, {"item": "stability re-runs on 20 calls (Stages 1 and 2)", "USD": round(runs, 2)}]))
nb.md(f"One full pass of Stages 1 and 2 over 100 calls cost about **${s1 + s2:.2f}** on the chosen model (Stage 0 adds a few cents). Cost scales roughly linearly with calls; the stated assumption is that transcripts of similar length are fed one at a time.")

item,USD
"bake-off (4 models, 475 calls)",1.63
Stage 1 over 100 calls,0.30
Stage 2 over 100 calls,0.61
stability re-runs on 20 calls (Stages 1 and 2),0.14


One full pass of Stages 1 and 2 over 100 calls cost about **$0.91** on the chosen model (Stage 0 adds a few cents). Cost scales roughly linearly with calls; the stated assumption is that transcripts of similar length are fed one at a time.

## Limitations, stated up front

1. **n = 100 calls from one fictional company that look synthetic**; no outcome labels. Findings are directional.
2. **No gold labels.** Agreement with the provided labels is consistency with an upstream AI output. The candidate's own hand-checks are the only human validation (I cannot confirm I stayed blind to the model answers), and they are small (15 and 20 items); Stage 0's review covered 15 items.
3. **Taxonomy and anchors were finalised under delegation (D8)** from an LLM-proposed draft built on the same calls they are applied to.
4. **The Other bucket (0%) and drift detection are untested** in this data.
5. **Incident flag is an LLM judgment, partly definitional; in the candidate's 20-call check it missed 3 and never over-flagged, so 42 is probably a floor** (touches vs core are reported separately); **call dates are partly inconsistent** with content.
6. **Model bake-off:** 15-call sample on draft prompts, the same data we validate on; the decision-rule flaw above.
7. **Reproducibility rests on the cache,** not on the model being deterministic. Stage 0's cache keys come from the bake-off runner (notebook 01).

## What a production version would add

- **Taxonomy versioning and drift monitoring:** a growing *Other* share or falling classification confidence triggers a revision, a version bump and a full re-run.
- **A gold set:** a few hundred human-labelled calls, refreshed over time, so accuracy (not just consistency) is measured; inter-reviewer agreement.
- **Tracing and cost monitoring:** an optional tracing hook is already in `src/ti/llm.py` (a no-op); self-hosted Langfuse would plug into it. It was not used for the original run and is not claimed for it.
- **Incremental runs:** classify only new calls against the frozen taxonomy; re-score everything only when anchors, prompt or model change.
- **A unified stage-0 code path** so the live runner and the frozen run share one cache key.
- **Outcome data** (renewals, churn) to validate the account-risk list.